### Đồ án cuối kỳ: Phân loại tin nhắn rác (SMS Spam Classification)
# "LSTM Master"

## 0. Cài đặt môi trường & Import thư viện

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf

# Cố định seed để kết quả có thể tái lập giữa các lần chạy
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 1. Chuẩn bị dữ liệu (kế thừa nguyên logic tiền xử lý của Thành viên 1)

In [ ]:
#Đọc dữ liệu
df = pd.read_csv('sms+spam+collection/SMSSpamCollection', sep='\t', header=None, names=['Label', 'Text'])
df['Label'] = df['Label'].map({'ham': 0, 'spam': 1})

#Làm sạch văn bản
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

df['Clean_Text'] = df['Text'].apply(clean_text)

#Tokenize
max_words = 10000
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['Clean_Text'])
sequences = tokenizer.texts_to_sequences(df['Clean_Text'])

#Padding + Train/Test split
max_len = 100
X_padded = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')
Y = df['Label'].values

X_train, X_test, Y_train, Y_test = train_test_split(
    X_padded, Y, test_size=0.2, random_state=SEED
)

print('Số tin nhắn để HỌC (Train):', len(X_train))
print('Số tin nhắn để THI (Test):', len(X_test))

## 2. Xây dựng mô hình LSTM

Kiến trúc: **Embedding → LSTM → (Dropout tùy chọn) → Dense (sigmoid)**.
Hàm được viết tổng quát để tái sử dụng cho cả 3 cấu hình thực nghiệm, tránh lặp code.

In [ ]:
embedding_dim = 64  # số chiều vector nhúng cho mỗi từ

def build_lstm_model(lstm_units=32, use_dropout=False, dropout_rate=0.5):
    """
    Khởi tạo mô hình LSTM để phân loại SMS Spam/Ham.

    Tham số
    ----------
    lstm_units : int
        Số đơn vị (units) của lớp LSTM.
    use_dropout : bool
        Có thêm lớp Dropout sau LSTM để chống overfitting hay không.
    dropout_rate : float
        Tỉ lệ Dropout (chỉ dùng khi use_dropout=True).

    Trả về
    -------
    tf.keras.Model đã compile, sẵn sàng để .fit()
    """
    model = Sequential([
        Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
        LSTM(lstm_units),
        *([Dropout(dropout_rate)] if use_dropout else []),
        Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

## 3. Thiết kế thực nghiệm: huấn luyện 3 cấu hình khác nhau

| Cấu hình | LSTM units | Dropout |
|---|---|---|
| Lần 1 | 32 | Không |
| Lần 2 | 64 | Không |
| Lần 3 | 64 | Có (0.5) |

**Tối ưu huấn luyện:** dùng `EarlyStopping` (theo dõi `val_loss`, dừng sớm nếu không cải thiện sau 3 epoch, tự khôi phục trọng số tốt nhất). Việc này vừa tiết kiệm thời gian chạy, vừa tránh huấn luyện dư epoch gây overfitting — giúp việc so sánh 3 cấu hình công bằng và nhanh hơn.

In [ ]:
configs = [
    {"name": "LSTM_32units",         "lstm_units": 32, "use_dropout": False, "dropout_rate": 0.0},
    {"name": "LSTM_64units",         "lstm_units": 64, "use_dropout": False, "dropout_rate": 0.0},
    {"name": "LSTM_64units_Dropout", "lstm_units": 64, "use_dropout": True,  "dropout_rate": 0.5},
]

MAX_EPOCHS = 15
BATCH_SIZE = 32

early_stop = EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

histories, models = {}, {}

for cfg in configs:
    print(f"\n=== Đang huấn luyện: {cfg['name']} ===")
    model = build_lstm_model(
        lstm_units=cfg['lstm_units'],
        use_dropout=cfg['use_dropout'],
        dropout_rate=cfg['dropout_rate'],
    )
    history = model.fit(
        X_train, Y_train,
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=1,
    )
    models[cfg['name']] = model
    histories[cfg['name']] = history
    print(f"--> Dừng ở epoch {len(history.history['loss'])} (EarlyStopping)")

## 4. Trực quan hóa Loss/Accuracy qua từng Epoch

In [ ]:
def plot_history(history, name):
    """Vẽ 2 biểu đồ Loss và Accuracy (Train vs Validation) cho 1 cấu hình."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Validation Loss')
    axes[0].set_title(f'{name} - Loss qua từng Epoch')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()

    axes[1].plot(history.history['accuracy'], label='Train Accuracy')
    axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy')
    axes[1].set_title(f'{name} - Accuracy qua từng Epoch')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend()

    plt.tight_layout()
    plt.show()

for name, history in histories.items():
    plot_history(history, name)

## 5. Bảng so sánh 3 cấu hình trên tập Test

In [ ]:
comparison_rows = []
for name, model in models.items():
    test_loss, test_acc = model.evaluate(X_test, Y_test, verbose=0)
    comparison_rows.append({
        "Cấu hình": name,
        "Số Epoch thực chạy": len(histories[name].history['loss']),
        "Test Loss": round(test_loss, 4),
        "Test Accuracy": round(test_acc, 4),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values('Test Accuracy', ascending=False)
comparison_df

## 6. Đánh giá chi tiết mô hình LSTM tốt nhất

In [ ]:
best_name = comparison_df.iloc[0]['Cấu hình']
best_model = models[best_name]
print(f"Mô hình LSTM tốt nhất theo Test Accuracy: {best_name}")

Y_pred_prob = best_model.predict(X_test)
Y_pred_lstm = (Y_pred_prob > 0.5).astype(int).flatten()

print("\nClassification Report (LSTM):")
print(classification_report(Y_test, Y_pred_lstm, target_names=['Ham', 'Spam']))

cm = confusion_matrix(Y_test, Y_pred_lstm)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Ham', 'Spam'])
disp.plot(cmap='Blues')
plt.title(f'Confusion Matrix - {best_name}')
plt.show()